### 4. PREDICT

In [38]:
from tensorflow.keras.models import load_model
import pickle
import numpy as np
import pandas as pd

In [39]:
### Load the trained model
model = load_model('model.h5')

## load the encoder and scaler
with open('onehot_encoder_geo.pkl', 'rb') as f:
    onehot_encoder_geo = pickle.load(f)

with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [40]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [41]:
input_data=pd.DataFrame([input_data])

In [42]:
# One-hot encode 'Geography'
geography_encoded = onehot_encoder_geo.transform(input_data[['Geography']])
feature_names = onehot_encoder_geo.get_feature_names_out(['Geography'])
geo_encoded_df = pd.DataFrame(geography_encoded, columns=feature_names)

geo_encoded_df



,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [43]:
## label encode 'Gender'
input_data['Gender'] = label_encoder_gender.transform(input_data['Gender'])

In [44]:
input_data=pd.concat([input_data.drop('Geography', axis=1).reset_index(drop=True), geo_encoded_df], axis=1)
input_data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [45]:
## Scaling the input data
scaled_input_data = scaler.transform(input_data)
scaled_input_data

array([[-0.52257777,  0.9148601 ,  0.09745106, -0.67664529, -0.25312915,
         0.80342994,  0.64176983,  0.96801137, -0.87309704,  0.97530483,
        -0.5658021 , -0.56965192]])

In [49]:
## predict the churn
prediction = model.predict(scaled_input_data)
prediction

1/1 [==============================] - 0s 21ms/step


array([[0.02329676]], dtype=float32)

In [50]:
prediction_proba=prediction[0][0]
prediction_proba

0.023296757

In [51]:
prediction_class = (prediction > 0.5).astype(int)
prediction_class

array([[0]])

In [52]:
if prediction_proba > 0.5:
    print(f"The customer is likely to churn with a probability of {prediction_proba:.2f}.")
else:
    print(f"The customer is unlikely to churn with a probability of {prediction_proba:.2f}.")

The customer is unlikely to churn with a probability of 0.02.
